In [1]:
import pandas as pd
import numpy as np


In [44]:
df_prev = pd.read_csv('previous_application.csv')
df_prev.head(5)

,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,...,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,...,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,...,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0
2,2523466,122040,Cash loans,15060.735,112500.0,136444.5,NaN,112500.0,TUESDAY,11,...,XNA,12.0,high,Cash X-Sell: high,365243.0,-271.0,59.0,365243.0,365243.0,1.0
3,2819243,176158,Cash loans,47041.335,450000.0,470790.0,NaN,450000.0,MONDAY,7,...,XNA,12.0,middle,Cash X-Sell: middle,365243.0,-482.0,-152.0,-182.0,-177.0,1.0
4,1784265,202054,Cash loans,31924.395,337500.0,404055.0,NaN,337500.0,THURSDAY,9,...,XNA,24.0,high,Cash Street: high,NaN,NaN,NaN,NaN,NaN,NaN


In [45]:
result = df_prev[df_prev["SK_ID_CURR"] == 427000]["AMT_ANNUITY"].mean()
print(result)

2090.07


In [46]:
#  выбираем только одобренные заявки
approved_loans = df_prev[df_prev['NAME_CONTRACT_STATUS'] == 'Approved']

# Фильтрация активных кредитов: DAYS_LAST_DUE != 365243
active_approved = approved_loans[approved_loans['DAYS_LAST_DUE'] != 365243]


count_active_approved = len(active_approved)

print(f"Количество активных одобренных кредитов: {count_active_approved}")

Количество активных одобренных кредитов: 229483


In [49]:
df_app = pd.read_csv("application_train3.csv")
df_app.head(3)

,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
2,100004,0,Revolving loans,M,Y,Y,0,67500.0,135000.0,6750.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [54]:
client_id = 456254

# 2. Фильтрация активных одобренных кредитов для клиента 456254
client_active_loans = df_prev[
    (df_prev['SK_ID_CURR'] == client_id) &
    (df_prev['NAME_CONTRACT_STATUS'] == 'Approved') &
    (df_prev['DAYS_LAST_DUE'] != 365243)
]

# 3. Суммирование платежей по активным кредитам
total_annuity = client_active_loans['AMT_ANNUITY'].sum()

# 4. Получение зарплаты клиента
client_income_row = df_app[df_app['SK_ID_CURR'] == client_id]

if client_income_row.empty:
    print(f"Ошибка: клиент с ID {client_id} не найден в данных о зарплатах")
else:
    client_income = client_income_row['AMT_INCOME_TOTAL'].iloc[0]
    
    # 5. Расчёт отношения с проверкой на ноль
    if client_income == 0:
        result = None
        print(f"Ошибка: зарплата клиента {client_id} равна 0, деление невозможно")
    else:
        ratio = total_annuity / client_income
        result = round(ratio, 4)  # округление до 4 знаков после запятой
    
    # 6. Вывод результата
    print(f"Результаты для клиента {client_id}:")
    print(f"  Суммарные платежи по активным кредитам: {total_annuity}")
    print(f"  Зарплата (AMT_INCOME_TOTAL): {client_income}")
    if result is not None:
        print(f"  Отношение платежей к зарплате: {result}")
   

Результаты для клиента 456254:
  Суммарные платежи по активным кредитам: 0.0
  Зарплата (AMT_INCOME_TOTAL): 171000.0
  Отношение платежей к зарплате: 0.0
